# Baseline Window Classifier (`SignNet` + Rule-Based `SignDecider`)

This notebook documents the **baseline (`OLD`) pipeline** and explains why the project transitioned from frame-window classification (`SignNet` + `recognizer.py`) to end-to-end sequence modeling (`SeqNet` + CTC + `stream_decoder.py`) for continuous fingerspelling (`ABC` mode).

## Architecture Comparison
| Pipeline | Model | Decoding Strategy | Strengths | Limitations on Continuous Words |
| :--- | :--- | :--- | :--- | :--- |
| **OLD (Baseline)** | `SignNet` (`src/model.py`) | Sliding window classification + dwell/cooldown rules (`src/recognizer.py`) | Strong on static single signs, `123`, and isolated `WORDS` mode | Struggles with rapid transitions, double letters (`ALL`, `BOOK`, `COFFEE`), and variable signing speed (73.8% CER on `words_test`) |
| **CTC (Current v6)** | `SeqNet` (`src/ctc_model.py`) | 4-second temporal sequence logits + CTC alignment + `StreamingDecoder` | Learns letter transitions and timing directly; handles double letters (`LOVE`, `TEA`, `QUICK`) | Requires sequence training & real-word fine-tuning (34.5% CER, 15.4% exact word on `words_test`) |

In [ ]:
from pathlib import Path
import sys

# Resolve project root whether launched from repo root or inside notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root : {PROJECT_ROOT}")
print(f"src path     : {SRC_DIR} (exists={SRC_DIR.exists()})")

## 1. Inspect Baseline `SignNet` Architecture & Rules
Load `src/model.py` and `src/recognizer.py` to inspect how `SignDecider` filters predictions in non-CTC mode.

In [ ]:
import inspect
import model as baseline_model
import recognizer

print("--- SignNet Definition ---")
print(inspect.getsource(baseline_model.SignNet))

print("--- SignDecider Init Signature ---")
print(inspect.signature(recognizer.SignDecider.__init__))

## 2. Benchmark Comparison on `data/words_test`
If the legacy `asl_model.pth` checkpoint is available locally, run `scripts/eval_words.py` to compare `OLD` vs `CTC (asl_ctc_v6.pth)` side by side on the 91 real test takes.

In [ ]:
import subprocess

ctc_ckpt = PROJECT_ROOT / "trained_models" / "ctc" / "asl_ctc_v6.pth"
old_ckpt = PROJECT_ROOT / "asl_model.pth"  # Optional local baseline checkpoint

if old_ckpt.exists():
    cmd = [
        sys.executable,
        str(SCRIPTS_DIR / "eval_words.py"),
        "--words-dir", str(PROJECT_ROOT / "data" / "words_test"),
        "--ctc", str(ctc_ckpt),
        "--old", str(old_ckpt),
    ]
    res = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
    print(res.stdout)
else:
    print("Legacy baseline checkpoint 'asl_model.pth' not in root (cleaned up).")
    print("Recorded benchmark on data/words_test (91 takes, 30 words, signer ['leyu_test']):")
    print("  OLD (SignNet + rules) : CER 73.8% | Exact-word accuracy  1.1%")
    print("  CTC (asl_ctc_v6.pth)  : CER 34.5% | Exact-word accuracy 15.4%")